# 08 时间扩展 A*：状态 (点, 时刻)


本课按百科条目写。把第 07 课的表接到 A\* 上：状态从「点」变成「点×时间」。调度台在最后。

---

## 1. 定义

**时间扩展 A\***（time-expanded A\*，时空 A\*）把时间离散进状态。状态

\[
s=(v,t)
\]

动作（本课一步一拍、边权即耗时）：

- **移动**：到空间邻居 \(v'\)，\(t'=t+1\)
- **等待**：\(v'=v\)，\(t'=t+1\)

若占用表禁止 \((v',t')\) 的顶点，或禁止边 \((v,v',t)\)，则该动作非法。

- 输入：图、起终点、`ReservationTable`、智能体名、horizon T。
- 输出：点列（相邻重复 = 等待），代价含等待。

WHCA 窗口规划会设 `require_goal=False`：到不了终点就返回窗口内离终点最近的前缀。

---

## 2. 和普通 A* 的关系

| | 普通 A\* | 时间扩展 A\* |
|--|--|--|
| 状态 | 点 | (点, t) |
| 等待 | 无（原地边会成自环且无收益） | 一等动作 |
| closed | 每个点一次 | 每个 (点,t) 一次 |
| h | 空间启发 | 仍用空间启发（可采纳：等待只让 g 变大） |
| 规模 | \(O(V)\) | \(O(V\cdot T)\) |

空占用表、允许等待时，最优代价 = 空间最短路（没有人逼你等）。有人挡路时代价 ≥ 空间最短路。

---

## 3. 不变量

- 返回路径上每个 `(path[t], t)` 对该 agent `vertex_free`。
- 相邻若移动，则离开拍 `edge_free(path[t], path[t+1], t)`。
- 第一次弹出终点时 g 最优（h 一致、非负边权、关闭后不重开）。
- `t` 超过 horizon 不再扩展，防止无限等。

---

## 4. 伪代码（对照下面 `time_astar_mini`）

```
g[s,0] ← 0
push (h(s), 0, s, 0)          # 键 f，然后 g, node, t
while heap:
    (f, cost, u, t) ← pop
    if (u,t) 已关闭: continue
    关闭 (u,t)
    if u = goal: 回溯点列
    if t ≥ horizon: continue
    for (v, w) in {(u, wait_cost=1)} ∪ 邻居(u):
        t2 ← t+1
        if 顶点 (v,t2) 被占: continue
        if v≠u 且 边 (u,v,t) 被占: continue
        alt ← cost + w
        松弛 (v,t2)，push (alt + h(v), alt, v, t2)
```

回溯：`came[(v,t2)] = (u,t)`，抽出点列（可含重复）。

`require_goal=False`：全程记 h 最小的已关闭状态，失败时回溯它。

---

## 5. 手算

走廊 `A—C1—C2—C3—B`，每边 1。空间最短 A→B 四步。现把 C2 在 t=0,1,2 占住。

t=2 不能进 C2。一种合法策略：A(0)–C1(1)–C1(2 等待)–C2(3)–C3(4)–B(5)。C2 下标应 ≥ 3。

若不允许等待，可能宣告失败——其实等一拍就行。这是最常见的实现漏洞。


In [ ]:
import sys
from pathlib import Path
ROOT = Path.cwd() if (Path.cwd() / "lib").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))


## 6. 实现：状态 (node, t)，含等待


In [ ]:
import heapq
from lib.maps import corridor_headon
from lib.reservation import ReservationTable
from lib.search import astar


def rebuild(came, state):
    path = []
    cur = state
    while cur is not None:
        path.append(cur[0])
        cur = came[cur]
    path.reverse()
    return path


def time_astar_mini(graph, start, goal, table, agent="me", horizon=40, require_goal=True, verbose=True):
    heap = []
    g = {(start, 0): 0.0}
    came = {(start, 0): None}
    heapq.heappush(heap, (graph.heuristic(start, goal), 0.0, start, 0))
    seen = set()
    expanded = 0
    best_partial = None
    best_h = float("inf")
    while heap:
        _f, cost, node, t = heapq.heappop(heap)
        state = (node, t)
        if state in seen:
            continue
        seen.add(state)
        expanded += 1
        hn = graph.heuristic(node, goal)
        if hn < best_h or (hn == best_h and t > 0):
            best_h = hn
            best_partial = state
        if verbose and expanded <= 15:
            print(f"关闭 {node:3} t={t} g={cost:.0f} h={hn:.0f}")
        if node == goal:
            path = rebuild(came, state)
            if verbose:
                print("到达", path, "代价", cost, "扩展", expanded)
            return path, cost, expanded
        if t >= horizon:
            continue
        candidates = [(node, 1.0)] + list(graph.neighbors(node))
        for nxt, step_cost in candidates:
            nt = t + 1
            if not table.vertex_free(nxt, nt, agent):
                continue
            if nxt != node and not table.edge_free(node, nxt, t, agent):
                continue
            ns = (nxt, nt)
            ng = cost + step_cost
            if ng < g.get(ns, float("inf")):
                g[ns] = ng
                came[ns] = state
                heapq.heappush(heap, (ng + graph.heuristic(nxt, goal), ng, nxt, nt))
    if not require_goal and best_partial is not None:
        path = rebuild(came, best_partial)
        return path, g[best_partial], expanded
    if verbose:
        print("失败，扩展", expanded)
    return None, float("inf"), expanded


g = corridor_headon()
print("空间最短", astar(g, "A", "B", record=False).path)
table = ReservationTable()
for t in range(3):
    table.occupy("C2", t, "blocker")
path, cost, n_exp = time_astar_mini(g, "A", "B", table)
print("C2 出现的下标（应>=3）", [i for i, n in enumerate(path or []) if n == "C2"])


## 7. 逐行

| 行 | 作用 |
|----|------|
| 状态 `(node,t)` | 同一点不同时刻是不同状态 |
| `candidates = [(node,1)] + neighbors` | 等待边权 1 |
| `vertex_free(nxt, t+1)` | 进入下一拍的格子 |
| `edge_free(node, nxt, t)` | **离开的这一拍** 占用边 |
| `seen` | 关闭 (node,t)，一致 h 下不必重开 |
| `horizon` | 防止一直等 |
| `require_goal=False` | 窗口规划返回 best_partial |

路径里相邻相同 ⇒ 等待。


In [ ]:
from lib.algos.temporal import time_astar

r = time_astar(g, "A", "B", table=table, agent="me")
print("库", r.path, "代价", r.cost, "扩展", r.expanded)

# 窗口：horizon 太短且必须到终点会失败；关掉 require_goal 则返回前缀
short = time_astar(g, "A", "B", table=table, agent="me", horizon=3, require_goal=True)
part = time_astar(g, "A", "B", table=table, agent="me", horizon=3, require_goal=False)
print("horizon=3 必须到终点 found", short.found)
print("horizon=3 允许部分", part.found, part.path, part.extra)


## 常见 bug

1. 忘记等待：被挡就失败。
2. closed 只用空间点：t=1 走过 C1，t=3 再也不能进 C1。
3. 边冲突检查用 `t+1` 而不是离开拍 `t`。
4. 不等长路径比较冲突时没 pad（那是多车层；单车这里只要自己的 (v,t) 合法）。
5. horizon 小于最短步数且 `require_goal=True` ⇒ 全失败。WHCA 必须关 require_goal。

## 条目小结

| 项目 | 内容 |
|------|------|
| 状态 | (点, t) |
| 动作 | 走或等，都 +1 拍 |
| 规模 | O(V T) |
| 保证 | 表合法前提下最优 |
| 下一课 | 把时刻收成安全区间，状态更少 |

## 练习

1. 不许等待（candidates 去掉原地），这张占 C2@0..2 的走廊还找不找得到路？
2. 自己占的格 `vertex_free` 为真。若 blocker 也叫 `me`，会发生什么？
3. 画出 (C1,t) 哪些 t 会关闭。等一拍对应哪次关闭？


In [ ]:
from lib.studio.widget import PlannerStudio
from lib.algos.temporal import time_astar

r = time_astar(g, "A", "B", table=table, agent="me")
studio = PlannerStudio()
studio.show_search(g.to_studio(), r, "time A*")
studio
